# 00. Can we trust every model update?

Implement a coordinate-wise median, then compare it with FedAvg on the three
deployed SuperNodes. Training still happens on those nodes; your aggregation
function runs at the coordinator. Wait for MNIST app **0.1.1** to be released
before starting. Complete [setup](../docs/setup.md) first.

See the [exercise guide](../docs/aggregation.md) for the full contract.


In [ ]:
from pathlib import Path
import sys
import numpy as np

ROOT = next(
    p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "notebooks/workshop.py").is_file() and (p / "configs/aggregation.toml").is_file()
)
sys.path.insert(0, str(ROOT / "notebooks"))
from workshop import aggregation_student, run_training

RUN_ON_FEDERATION = False  # Enable after login, membership and the local checks.
FEDERATION = "@charles/flta"


## Predict, then implement

For scalar updates `[1, 2, 100]`, predict the mean and median. What if the large
value is a valid contribution from a client with different data?

Open `work/paris-mnist-fl/mnist_fl/aggregation_student.py`. Complete
`coordinate_median(updates)`: input shape `(clients, parameters)`, output shape
`(parameters,)`. Treat each coordinate independently and preserve the input.
The supplied adapter handles model conversion and adds the result to the
starting global model. Save the file and rerun the check cell after each edit.


In [ ]:
student = aggregation_student(ROOT)
updates = np.array([[1., -4., 20.], [2., -3., 0.], [100., 7., -10.]])
before = updates.copy()
result = student.coordinate_median(updates)
assert isinstance(result, np.ndarray) and result.shape == (3,)
np.testing.assert_array_equal(result, [2., -3., 0.])
np.testing.assert_array_equal(updates, before)
np.testing.assert_allclose(student.coordinate_median(updates + 17.), result + 17.)
even = np.array([[1., 3.], [2., 4.], [5., 7.], [9., 11.]])
np.testing.assert_array_equal(student.coordinate_median(even), [3.5, 5.5])
print("Coordinate, sign, translation and even-count checks passed")


## Choose a condition

Choose **one** with your group, then compare the class's results:

| `CASE` | Data | Returned updates |
| --- | --- | --- |
| `iid-clean` | IID | All honest |
| `iid-scaled` | IID | Partition 2 scales its update by 20 |
| `label-skew-clean` | Label-skewed | All honest |

The fault returns `global + 20 * (local - global)` for one partition. It is
per-run and does not change the shared SuperNodes. All cases use the same model,
seed, learning rate and two training rounds; evaluation uses 900 test images.
Record your prediction before running. The next two cells preview commands
until you enable `RUN_ON_FEDERATION` in the setup cell and rerun it.


In [ ]:
CASE = "iid-scaled"
mean_run = run_training(ROOT, "fedavg", CASE, execute=RUN_ON_FEDERATION, federation=FEDERATION)


In [ ]:
median_run = run_training(ROOT, "median", CASE, execute=RUN_ON_FEDERATION, federation=FEDERATION)


In [ ]:
if mean_run is None or median_run is None:
    print("Comparison pending: complete both deployed runs")
else:
    mean_config = {k: v for k, v in mean_run["run_config"].items() if k != "aggregation"}
    median_config = {k: v for k, v in median_run["run_config"].items() if k != "aggregation"}
    assert mean_config == median_config, "Keep all settings except aggregation identical"
    print("rule | test accuracy | test loss | last update norm | distance to FedAvg")
    for run in [mean_run, median_run]:
        s = run["summary"]
        d = s["diagnostics"][-1]
        print(f"{s['aggregation']:6} | {s['accuracy']:.4f} | {s['loss']:.4f} | "
              f"{d['aggregate-update-norm']:.4f} | {d['distance-to-mean']:.4f}")
        print("run:", s["fl_run_id"])


## Compare explanations

| Observation | Your result |
| --- | --- |
| Condition and prediction | |
| FedAvg run ID, accuracy and loss | |
| Median run ID, accuracy and loss | |
| What the class observed under the other conditions | |
| One limitation of this experiment | |

Why might robust aggregation discard useful information from honest clients?
Does a large update establish that a client is faulty? With three clients, how
would symmetric trimming differ from a median? Would removing only the smallest
value be appropriate for coordinates that may be negative?

The distance-to-FedAvg diagnostic is a difference between rules, not an error
against ground truth. No outcome is required to beat another on accuracy.
These short neural-network runs do not establish a general convergence or
Byzantine-robustness guarantee.

**Checkpoint:** correct coordinate-wise aggregation and two completed comparable
runs on the deployed federation, with an explanation of the observed trade-off.
If you finish early, try another condition or design a norm-clipping experiment.

Next: [digit counts](01_digit_counts.ipynb).
